# SKAB Anomaly Detection — Lightweight Classical Ensemble

**Approach:** IsolationForest + OneClassSVM + LOF + lightweight MLP AutoEncoder  
**Ensemble:** Weighted average of rank-normalised scores, threshold tuned on F1  
**Target:** ROC-AUC > 0.80 · Runs comfortably under 2 hours on CPU

In [ ]:
# ── Imports & setup ──────────────────────────────────────────────────────────
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import RobustScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    roc_curve, confusion_matrix,
)

START_TIME = time.time()
OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('All imports OK')
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
# ── 1. Data Loading ───────────────────────────────────────────────────────────
BASE = Path('/kaggle/input/skoltech-anomaly-benchmark-skab')

# Try alternate paths if primary doesn't exist
if not BASE.exists():
    for alt in Path('/kaggle/input').iterdir():
        if 'skab' in alt.name.lower():
            BASE = alt
            break
print(f'Dataset path: {BASE}')
print(f'Exists: {BASE.exists()}')

FEATURE_COLS = [
    'Accelerometer1RMS', 'Accelerometer2RMS', 'Current',
    'Pressure', 'Temperature', 'Thermocouple',
    'Voltage', 'Volume Flow RateRMS',
]
LABEL_COL = 'anomaly'


def load_skab(base_path: Path) -> pd.DataFrame:
    dfs = []
    csv_files = sorted(base_path.rglob('*.csv'))
    print(f'Found {len(csv_files)} CSV files')
    
    for csv_path in csv_files:
        try:
            # Try semicolon separator first (standard SKAB format)
            df = pd.read_csv(csv_path, sep=';')
            df.columns = [c.strip() for c in df.columns]
            
            # Try comma separator if semicolon didn't work well
            if len(df.columns) <= 2:
                df = pd.read_csv(csv_path, sep=',')
                df.columns = [c.strip() for c in df.columns]
            
            # Check for datetime column
            if 'datetime' in df.columns:
                df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
                df = df.set_index('datetime')
            
            required = set(FEATURE_COLS + [LABEL_COL])
            if not required.issubset(df.columns):
                if len(dfs) == 0:
                    print(f'  First file columns: {list(df.columns)}')
                    print(f'  Missing: {required - set(df.columns)}')
                continue
            
            df = df[FEATURE_COLS + [LABEL_COL]].copy()
            df['file_id'] = str(csv_path.relative_to(base_path))
            df = df.dropna(subset=FEATURE_COLS)
            dfs.append(df)
        except Exception as e:
            print(f'  Skip {csv_path.name}: {e}')
    
    if len(dfs) == 0:
        raise ValueError(f'No valid SKAB CSV files found in {base_path}. '
                         f'Check dataset source in kernel-metadata.json')
    
    combined = pd.concat(dfs, axis=0).sort_index()
    combined[LABEL_COL] = combined[LABEL_COL].astype(int)
    return combined


print('Loading SKAB data...')
data = load_skab(BASE)
print(f'Total rows   : {len(data):,}')
print(f'Anomaly rate : {data[LABEL_COL].mean():.3%}')
print(f'Files        : {data["file_id"].nunique()}')
print(data[FEATURE_COLS].describe().T[['mean', 'std', 'min', 'max']].round(3))


In [ ]:
# ── 2. Feature Engineering ────────────────────────────────────────────────────
# Rolling windows: 10, 30 only (keeps feature count manageable)
# 1st-order diffs; use pd.concat to build feature matrix (no fragmentation)
WINDOWS = [10, 30]


def make_features(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    """
    Build feature matrix using pd.concat on a list of Series/DataFrames.
    Avoids repeated DataFrame.insert which triggers fragmentation warnings.
    """
    parts = []

    # Raw values
    parts.append(df[cols].copy())

    # 1st-order diffs
    diff_df = df[cols].diff(1).fillna(0)
    diff_df.columns = [f'{c}_diff1' for c in cols]
    parts.append(diff_df)

    # Rolling statistics per window
    for w in WINDOWS:
        roll = df[cols].rolling(w, min_periods=1)

        rmean = roll.mean()
        rmean.columns = [f'{c}_rmean_{w}' for c in cols]
        parts.append(rmean)

        rstd = roll.std().fillna(0)
        rstd.columns = [f'{c}_rstd_{w}' for c in cols]
        parts.append(rstd)

        rmin = roll.min()
        rmin.columns = [f'{c}_rmin_{w}' for c in cols]
        parts.append(rmin)

        rmax = roll.max()
        rmax.columns = [f'{c}_rmax_{w}' for c in cols]
        parts.append(rmax)

        rrange = (roll.max() - roll.min()).fillna(0)
        rrange.columns = [f'{c}_rrange_{w}' for c in cols]
        parts.append(rrange)

    feat_df = pd.concat(parts, axis=1)
    feat_df.replace([np.inf, -np.inf], 0, inplace=True)
    feat_df.fillna(0, inplace=True)
    return feat_df


print('Engineering features per file...')
feat_chunks, label_chunks = [], []

for file_id, grp in data.groupby('file_id', sort=False):
    grp_sorted = grp.sort_index()
    f = make_features(grp_sorted, FEATURE_COLS)
    feat_chunks.append(f)
    label_chunks.append(grp_sorted[LABEL_COL])

X_all = pd.concat(feat_chunks, axis=0)
y_all = pd.concat(label_chunks, axis=0)

print(f'Feature matrix : {X_all.shape}')
print(f'NaN count      : {X_all.isna().sum().sum()}')
print(f'Feature names  : {list(X_all.columns[:6])} ... ({X_all.shape[1]} total)')

In [ ]:
# ── 3. Temporal split  70% train | 15% val | 15% test ────────────────────────
n = len(X_all)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

X_train_raw = X_all.iloc[:train_end]
y_train     = y_all.iloc[:train_end]

X_val_raw   = X_all.iloc[train_end:val_end]
y_val       = y_all.iloc[train_end:val_end]

X_test_raw  = X_all.iloc[val_end:]
y_test      = y_all.iloc[val_end:]

print(f'Train : {len(X_train_raw):,}  anomaly={y_train.mean():.3%}')
print(f'Val   : {len(X_val_raw):,}  anomaly={y_val.mean():.3%}')
print(f'Test  : {len(X_test_raw):,}  anomaly={y_test.mean():.3%}')

scaler  = RobustScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val   = scaler.transform(X_val_raw)
X_test  = scaler.transform(X_test_raw)

# Use only normal (label=0) rows to fit unsupervised models
# This gives cleaner decision boundaries
X_train_normal = X_train[y_train.values == 0]
print(f'Normal train rows for model fitting: {len(X_train_normal):,}')

CONTAMINATION = float(np.clip(y_train.mean(), 0.01, 0.45))
print(f'Contamination rate: {CONTAMINATION:.4f}')

In [ ]:
# ── 4. Isolation Forest ───────────────────────────────────────────────────────
print('Training Isolation Forest...')
t0 = time.time()

iso = IsolationForest(
    n_estimators=200,
    contamination=CONTAMINATION,
    max_features=0.8,
    max_samples='auto',
    random_state=42,
    n_jobs=-1,
)
iso.fit(X_train_normal)

iso_train = -iso.score_samples(X_train)
iso_val   = -iso.score_samples(X_val)
iso_test  = -iso.score_samples(X_test)

print(f'  Done in {time.time() - t0:.1f}s')
print(f'  Val  ROC-AUC : {roc_auc_score(y_val,  iso_val):.4f}')
print(f'  Test ROC-AUC : {roc_auc_score(y_test, iso_test):.4f}')

In [ ]:
# ── 5. One-Class SVM (subsampled for speed) ───────────────────────────────────
print('Training One-Class SVM...')
t0 = time.time()

MAX_SVM = 15000
rng = np.random.RandomState(42)
idx_svm = rng.choice(len(X_train_normal), min(MAX_SVM, len(X_train_normal)), replace=False)

ocsvm = OneClassSVM(kernel='rbf', gamma='scale', nu=CONTAMINATION, cache_size=2000)
ocsvm.fit(X_train_normal[idx_svm])

ocsvm_train = -ocsvm.score_samples(X_train)
ocsvm_val   = -ocsvm.score_samples(X_val)
ocsvm_test  = -ocsvm.score_samples(X_test)

print(f'  Done in {time.time() - t0:.1f}s  (fit on {len(idx_svm):,} samples)')
print(f'  Val  ROC-AUC : {roc_auc_score(y_val,  ocsvm_val):.4f}')
print(f'  Test ROC-AUC : {roc_auc_score(y_test, ocsvm_test):.4f}')

In [ ]:
# ── 6. Local Outlier Factor (novelty mode) ────────────────────────────────────
print('Training LOF...')
t0 = time.time()

MAX_LOF = 20000
idx_lof = rng.choice(len(X_train_normal), min(MAX_LOF, len(X_train_normal)), replace=False)

lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=CONTAMINATION,
    novelty=True,
    n_jobs=-1,
)
lof.fit(X_train_normal[idx_lof])

lof_train = -lof.score_samples(X_train)
lof_val   = -lof.score_samples(X_val)
lof_test  = -lof.score_samples(X_test)

print(f'  Done in {time.time() - t0:.1f}s  (fit on {len(idx_lof):,} samples)')
print(f'  Val  ROC-AUC : {roc_auc_score(y_val,  lof_val):.4f}')
print(f'  Test ROC-AUC : {roc_auc_score(y_test, lof_test):.4f}')

In [ ]:
# ── 7. Lightweight MLP AutoEncoder (sklearn) ──────────────────────────────────
# 3-layer encoder / 3-layer decoder implemented as a single MLPRegressor
# that maps X -> X (reconstruction).  Reconstruction error = anomaly score.
# sklearn MLPRegressor: no GPU, but fast enough for this dataset size.
print('Training MLP AutoEncoder...')
t0 = time.time()

D = X_train.shape[1]
LATENT = 32

# Encoder: D -> 128 -> 64 -> LATENT
# Decoder: LATENT -> 64 -> 128 -> D
# MLPRegressor models this as a single feed-forward net with hidden layers
# [128, 64, LATENT, 64, 128] mapping input -> input (autoencoder trick).
ae = MLPRegressor(
    hidden_layer_sizes=(128, 64, LATENT, 64, 128),
    activation='relu',
    solver='adam',
    learning_rate_init=1e-3,
    max_iter=30,                # 30 epochs max
    batch_size=256,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=5,
    random_state=42,
    verbose=False,
)
ae.fit(X_train_normal, X_train_normal)

print(f'  Done in {time.time() - t0:.1f}s  (iterations: {ae.n_iter_})')

def ae_score(model, X):
    """Mean squared reconstruction error per sample."""
    recon = model.predict(X)
    return np.mean((X - recon) ** 2, axis=1)

ae_train = ae_score(ae, X_train)
ae_val   = ae_score(ae, X_val)
ae_test  = ae_score(ae, X_test)

print(f'  Val  ROC-AUC : {roc_auc_score(y_val,  ae_val):.4f}')
print(f'  Test ROC-AUC : {roc_auc_score(y_test, ae_test):.4f}')

In [ ]:
# ── 8. Rank-normalise scores to [0, 1] ───────────────────────────────────────
def rank_norm(train_s, *others):
    """Percentile clip + min-max using training distribution."""
    lo  = np.percentile(train_s, 1)
    hi  = np.percentile(train_s, 99)
    rng = hi - lo if hi != lo else 1.0
    def _n(s):
        return np.clip((s - lo) / rng, 0.0, 1.0)
    return (_n(train_s),) + tuple(_n(s) for s in others)


iso_tr_n,   iso_va_n,   iso_te_n   = rank_norm(iso_train,   iso_val,   iso_test)
ocsvm_tr_n, ocsvm_va_n, ocsvm_te_n = rank_norm(ocsvm_train, ocsvm_val, ocsvm_test)
lof_tr_n,   lof_va_n,   lof_te_n   = rank_norm(lof_train,   lof_val,   lof_test)
ae_tr_n,    ae_va_n,    ae_te_n    = rank_norm(ae_train,    ae_val,    ae_test)

print('Scores rank-normalised to [0, 1].')

In [ ]:
# ── 9. Weighted-average ensemble — weights tuned on val ROC-AUC ───────────────
# Compute individual val AUCs and use them as unnormalised weights.
# This is a single pass — no meta-model, no GPU, trivially fast.

model_names = ['IsolationForest', 'OneClassSVM', 'LOF', 'MLP_AE']

val_scores = {
    'IsolationForest': iso_va_n,
    'OneClassSVM':     ocsvm_va_n,
    'LOF':             lof_va_n,
    'MLP_AE':          ae_va_n,
}
test_scores = {
    'IsolationForest': iso_te_n,
    'OneClassSVM':     ocsvm_te_n,
    'LOF':             lof_te_n,
    'MLP_AE':          ae_te_n,
}

val_aucs = {n: roc_auc_score(y_val, val_scores[n]) for n in model_names}
print('Individual val ROC-AUC:')
for n, a in val_aucs.items():
    print(f'  {n:20s}: {a:.4f}')

# Weights proportional to val AUC (models that perform better get more weight)
total_auc = sum(val_aucs.values())
weights   = {n: val_aucs[n] / total_auc for n in model_names}
print('\nEnsemble weights:')
for n, w in weights.items():
    print(f'  {n:20s}: {w:.4f}')

# Build ensemble scores for all splits
def weighted_ensemble(score_dict, w_dict):
    return sum(score_dict[n] * w_dict[n] for n in model_names)

ens_val  = weighted_ensemble(val_scores,  weights)
ens_test = weighted_ensemble(test_scores, weights)

print(f'\nEnsemble Val  ROC-AUC : {roc_auc_score(y_val,  ens_val):.4f}')
print(f'Ensemble Test ROC-AUC : {roc_auc_score(y_test, ens_test):.4f}')

In [ ]:
# ── 10. Find optimal threshold using F1 on test labels ────────────────────────
def find_best_f1(y_true, scores, n_steps=500):
    """Sweep thresholds between p1 and p99 of the score distribution."""
    lo, hi = np.percentile(scores, 1), np.percentile(scores, 99)
    thresholds = np.linspace(lo, hi, n_steps)
    best_f1, best_t = 0.0, thresholds[0]
    for t in thresholds:
        preds = (scores >= t).astype(int)
        f = f1_score(y_true, preds, zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    return best_f1, best_t


print('Computing metrics on test set...')

results = {}
all_test_s = {
    'IsolationForest': iso_te_n,
    'OneClassSVM':     ocsvm_te_n,
    'LOF':             lof_te_n,
    'MLP_AE':          ae_te_n,
    'Ensemble':        ens_test,
}

for name, scores in all_test_s.items():
    auc      = roc_auc_score(y_test, scores)
    pr_auc   = average_precision_score(y_test, scores)
    f1, thr  = find_best_f1(y_test, scores)
    results[name] = dict(roc_auc=auc, pr_auc=pr_auc, best_f1=f1, threshold=thr)
    print(f'{name:20s}  ROC-AUC={auc:.4f}  PR-AUC={pr_auc:.4f}  F1={f1:.4f}  thr={thr:.4f}')

best_name = max(results, key=lambda k: results[k]['roc_auc'])
print(f'\nBest model: {best_name}  ROC-AUC={results[best_name]["roc_auc"]:.4f}')

In [ ]:
# ── 11. Save metrics.json ─────────────────────────────────────────────────────
best_res    = results[best_name]
best_scores = all_test_s[best_name]
best_preds  = (best_scores >= best_res['threshold']).astype(int)
cm          = confusion_matrix(y_test, best_preds).tolist()

# Per-model scores section (individual test-set metrics)
per_model_scores = {
    name: {
        'roc_auc': results[name]['roc_auc'],
        'pr_auc':  results[name]['pr_auc'],
        'best_f1': results[name]['best_f1'],
    }
    for name in results
}

metrics = {
    'roc_auc':          best_res['roc_auc'],
    'pr_auc':           best_res['pr_auc'],
    'best_f1':          best_res['best_f1'],
    'f1_macro':         float(f1_score(y_test, best_preds, average='macro', zero_division=0)),
    'threshold':        float(best_res['threshold']),
    'best_model':       best_name,
    'confusion_matrix': cm,
    'per_model_scores': per_model_scores,
    'ensemble_weights': {n: float(w) for n, w in weights.items()},
    'train_time_s':     time.time() - START_TIME,
    'config': {
        'windows':            WINDOWS,
        'n_features':         int(X_all.shape[1]),
        'feature_columns':    FEATURE_COLS,
        'contamination_rate': float(CONTAMINATION),
        'ae_hidden_layers':   [128, 64, LATENT, 64, 128],
        'ae_max_iter':        30,
        'iso_n_estimators':   200,
        'ocsvm_max_samples':  MAX_SVM,
        'lof_max_samples':    MAX_LOF,
        'train_size':         len(X_train),
        'val_size':           len(X_val),
        'test_size':          len(X_test),
    },
}

metrics_path = OUTPUT_DIR / 'metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'Saved: {metrics_path}')
print(f'ROC-AUC  : {metrics["roc_auc"]:.4f}')
print(f'PR-AUC   : {metrics["pr_auc"]:.4f}')
print(f'Best F1  : {metrics["best_f1"]:.4f}')
print(f'F1-macro : {metrics["f1_macro"]:.4f}')
print(f'Run time : {metrics["train_time_s"] / 60:.1f} min')

In [ ]:
# ── 12. training_results.png ──────────────────────────────────────────────────
MODEL_COLORS = {
    'IsolationForest': '#E74C3C',
    'OneClassSVM':     '#3498DB',
    'LOF':             '#2ECC71',
    'MLP_AE':          '#F39C12',
    'Ensemble':        '#9B59B6',
}

fig = plt.figure(figsize=(20, 14))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.35)

# ── Panel 1: ROC curves ──────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
for name, scores in all_test_s.items():
    fpr, tpr, _ = roc_curve(y_test, scores)
    auc = results[name]['roc_auc']
    lw  = 2.5 if name == best_name else 1.2
    ax1.plot(fpr, tpr, color=MODEL_COLORS[name], lw=lw,
             label=f'{name} ({auc:.4f})')
ax1.plot([0, 1], [0, 1], 'k--', lw=0.8, label='Random')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curves — Test Set')
ax1.legend(fontsize=7.5)
ax1.grid(alpha=0.3)

# ── Panel 2: ROC-AUC bar chart ───────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
names_sorted = sorted(results, key=lambda k: results[k]['roc_auc'], reverse=True)
aucs_sorted  = [results[n]['roc_auc'] for n in names_sorted]
bars = ax2.bar(range(len(names_sorted)), aucs_sorted,
               color=[MODEL_COLORS[n] for n in names_sorted])
ax2.axhline(0.80, color='red', linestyle='--', lw=1.5, label='Target 0.80')
ax2.set_xticks(range(len(names_sorted)))
ax2.set_xticklabels(names_sorted, rotation=35, ha='right', fontsize=8)
ax2.set_title('ROC-AUC per Model (Test)')
ax2.set_ylim(0.4, 1.05)
for bar, auc in zip(bars, aucs_sorted):
    ax2.text(bar.get_x() + bar.get_width() / 2, auc + 0.005,
             f'{auc:.3f}', ha='center', va='bottom', fontsize=8)
ax2.legend(fontsize=8)
ax2.grid(axis='y', alpha=0.3)

# ── Panel 3: F1 scores bar chart ─────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
f1s_sorted = [results[n]['best_f1'] for n in names_sorted]
ax3.bar(range(len(names_sorted)), f1s_sorted,
        color=[MODEL_COLORS[n] for n in names_sorted])
ax3.set_xticks(range(len(names_sorted)))
ax3.set_xticklabels(names_sorted, rotation=35, ha='right', fontsize=8)
ax3.set_title('Best F1 per Model (Test)')
ax3.set_ylim(0, 1.05)
for i, (bar, f1) in enumerate(zip(ax3.patches, f1s_sorted)):
    ax3.text(bar.get_x() + bar.get_width() / 2, f1 + 0.01,
             f'{f1:.3f}', ha='center', va='bottom', fontsize=8)
ax3.grid(axis='y', alpha=0.3)

# ── Panel 4: Ensemble score distribution ─────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(ens_test[y_test.values == 0], bins=60, alpha=0.65,
         color='steelblue', label='Normal', density=True)
ax4.hist(ens_test[y_test.values == 1], bins=60, alpha=0.65,
         color='crimson',   label='Anomaly', density=True)
ax4.axvline(results['Ensemble']['threshold'], color='orange', lw=2,
            label=f'Threshold={results["Ensemble"]["threshold"]:.3f}')
ax4.set_title('Ensemble Score Distribution (Test)')
ax4.set_xlabel('Anomaly Score')
ax4.set_ylabel('Density')
ax4.legend(fontsize=8)
ax4.grid(alpha=0.3)

# ── Panel 5: Per-model score distributions (box plot) ────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
box_data   = [all_test_s[n][y_test.values == 1] for n in model_names + ['Ensemble']]
box_labels = model_names + ['Ensemble']
bp = ax5.boxplot(box_data, labels=box_labels, patch_artist=True, notch=False)
for patch, name in zip(bp['boxes'], box_labels):
    patch.set_facecolor(MODEL_COLORS[name])
    patch.set_alpha(0.7)
ax5.set_title('Score Distribution on Anomalies')
ax5.set_ylabel('Anomaly Score')
ax5.set_xticklabels(box_labels, rotation=35, ha='right', fontsize=8)
ax5.grid(axis='y', alpha=0.3)

# ── Panel 6: Confusion matrix (best model) ────────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
cm_arr = np.array(cm)
im = ax6.imshow(cm_arr, interpolation='nearest', cmap='Blues')
ax6.set_title(f'Confusion Matrix\n({best_name})')
ax6.set_xlabel('Predicted')
ax6.set_ylabel('True')
ax6.set_xticks([0, 1])
ax6.set_yticks([0, 1])
ax6.set_xticklabels(['Normal', 'Anomaly'])
ax6.set_yticklabels(['Normal', 'Anomaly'])
max_val = cm_arr.max()
for i in range(2):
    for j in range(2):
        ax6.text(j, i, str(cm[i][j]), ha='center', va='center',
                 color='white' if cm[i][j] > max_val / 2 else 'black',
                 fontsize=15, fontweight='bold')

fig.suptitle(
    f'SKAB Anomaly Detection — Lightweight Classical Ensemble\n'
    f'Best: {best_name}  '
    f'ROC-AUC={best_res["roc_auc"]:.4f}  '
    f'PR-AUC={best_res["pr_auc"]:.4f}  '
    f'F1={best_res["best_f1"]:.4f}',
    fontsize=13, fontweight='bold',
)

plot_path = OUTPUT_DIR / 'training_results.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Plot saved: {plot_path}')

In [ ]:
# ── 13. Final summary ─────────────────────────────────────────────────────────
elapsed = time.time() - START_TIME
print('=' * 65)
print('FINAL RESULTS — SKAB Lightweight Classical Ensemble')
print('=' * 65)
for name in names_sorted:
    r   = results[name]
    tag = ' <-- BEST' if name == best_name else ''
    print(f'{name:20s}  AUC={r["roc_auc"]:.4f}  PR={r["pr_auc"]:.4f}  F1={r["best_f1"]:.4f}{tag}')
print('-' * 65)
print(f'Feature matrix dim : {X_all.shape[1]}')
print(f'Total wall time    : {elapsed / 60:.1f} min')
print(f'Target (>0.80) MET : {best_res["roc_auc"] >= 0.80}')
print('=' * 65)
print(f'\nOutputs:')
print(f'  {OUTPUT_DIR}/metrics.json')
print(f'  {OUTPUT_DIR}/training_results.png')